# 11.2 TCP Client and Server

**Prerequisites:** 11.1 Networking Fundamentals, 12 threading (used here, explained there)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The seven calls that make a TCP server, and the three that make a client
- Running a server on a background thread so a notebook stays usable
- 🔴 `recv()` returning `b""` - the only way to know the peer has gone
- 🔴 **The framing problem**: TCP is a byte stream, not a message queue
- Three ways to frame messages, and when each is right
- `send()` vs `sendall()` - the partial-write trap
- `SO_REUSEADDR`, `TIME_WAIT`, and "Address already in use"
- `ConnectionResetError` and `BrokenPipeError` - what they actually mean

---

## The shape of it

```
   SERVER                                 CLIENT
   ─────────────────────────────          ──────────────────────
   s = socket()                           c = socket()
   s.setsockopt(SO_REUSEADDR, 1)
   s.bind((host, port))
   s.listen(backlog)
   conn, addr = s.accept()      <───────  c.connect((host, port))
        │  a NEW socket, just for this client
   conn.recv(n)                 <───────  c.sendall(data)
   conn.sendall(reply)          ───────>  c.recv(n)
   conn.close()                           c.close()
   s.close()
```

The step that catches people is `accept()`. It **blocks until a client arrives**, then returns a *brand-new socket* for that conversation. The listening socket goes straight back to listening. A server handling ten clients holds eleven sockets.

### Why we use a background thread

`accept()` blocking is fine in a script whose whole job is to serve. In a notebook it would freeze the cell forever. So every server here runs on a **daemon thread** with a **timeout on every blocking call**, and binds to **port 0** so re-running never hits "Address already in use".

### A small harness, used by every example below

`serve(handler)` starts a server on a background thread and hands back the address it landed on plus a way to stop it. The `accept()` timeout is what lets the loop notice the stop flag instead of blocking forever.

In [ ]:
import socket
import threading
import time


def serve(handler, *, accept_timeout=0.3, conn_timeout=3.0):
    """Run `handler(conn)` for each client, on a daemon thread.

    Returns (address, stop) - call stop() when finished.
    """
    ready = threading.Event()
    stop_flag = threading.Event()
    box = {}

    def loop():
        srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        # Without SO_REUSEADDR a just-closed port can sit in TIME_WAIT
        # and refuse to be rebound. See the section on that below.
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(("127.0.0.1", 0))          # 0 -> the OS picks a free port
        srv.listen(5)                       # 5 = backlog of pending connections
        srv.settimeout(accept_timeout)      # 🔴 so accept() cannot block forever
        box["address"] = srv.getsockname()
        ready.set()
        while not stop_flag.is_set():
            try:
                conn, _addr = srv.accept()
            except (TimeoutError, socket.timeout):
                continue                    # nobody arrived; check the flag again
            with conn:
                conn.settimeout(conn_timeout)
                try:
                    handler(conn)
                except (OSError, ConnectionError):
                    pass                    # a client that vanished is not our problem
        srv.close()

    thread = threading.Thread(target=loop, daemon=True)
    thread.start()
    ready.wait(timeout=5)

    def stop():
        stop_flag.set()
        thread.join(timeout=5)

    return box["address"], stop


print("harness ready")

## A first server: echo

The `hello world` of sockets. Read what the client sent, send it back.

`socket.create_connection((host, port), timeout=...)` is the client-side convenience function: it creates the socket, resolves the address, connects, and handles IPv4/IPv6 for you. Prefer it to `socket()` + `connect()`.

In [ ]:
def echo_handler(conn):
    data = conn.recv(4096)
    if data:
        conn.sendall(b"echo: " + data)


address, stop = serve(echo_handler)
print("server listening on", address)

# ---- the client ----
with socket.create_connection(address, timeout=3.0) as client:
    client.sendall(b"hello, sockets")
    reply = client.recv(4096)

print("client sent : b'hello, sockets'")
print("client got  :", reply)

stop()
print("server stopped")

### 🔴 `recv()` returning `b""` means the peer has closed

This is the one signal TCP gives you, and missing it is how people write loops that spin at 100% CPU forever.

| `recv()` returns | Means |
|---|---|
| `b"some bytes"` | data arrived |
| `b""` | **the peer closed cleanly** — never any more data |
| raises `TimeoutError` | nothing arrived within the timeout; the peer may still be there |
| raises `ConnectionResetError` | the peer vanished abruptly |

`b""` is end-of-stream, exactly like an empty read from a file in **8.1**. It is not an error and it will keep being returned forever if you ignore it.

In [ ]:
def until_closed(conn):
    """Read until the client goes away, counting what arrives."""
    chunks, reads = [], 0
    while True:
        data = conn.recv(4096)
        reads += 1
        if not data:                     # b'' -> the client closed
            break
        chunks.append(data)
    result["chunks"] = chunks
    result["reads"] = reads


result = {}
address, stop = serve(until_closed)

client = socket.create_connection(address, timeout=3.0)
client.sendall(b"one")
time.sleep(0.1)
client.sendall(b"two")
time.sleep(0.1)
client.close()                           # <- this is what produces b''
time.sleep(0.3)
stop()

print("chunks the server read:", result.get("chunks"))
print("recv() calls made     :", result.get("reads"))
print("  ^ the last one returned b'' and broke the loop")

---

# 🔴 The framing problem

This is the single most important idea in TCP programming, and the one most likely to produce a bug that only appears under load or across a real network.

> **TCP guarantees your bytes arrive, in order, exactly once.**
> **TCP guarantees nothing whatsoever about how they are grouped.**

Three calls to `sendall()` are not three messages. They are bytes appended to a stream. The receiver may see them as one read, or three, or seven — and it can differ between runs, between localhost and a real network, and between small and large payloads.

Causes include the kernel's send buffer, **Nagle's algorithm** (which deliberately coalesces small writes to avoid flooding the network with tiny packets), MTU-sized splitting, and retransmission.

The next cell sends `AAA`, `BBB`, `CCC` as three separate calls, waits, and reads once.

In [ ]:
def one_read(conn):
    time.sleep(0.4)                      # let everything arrive first
    result["got"] = conn.recv(4096)


result = {}
address, stop = serve(one_read)

with socket.create_connection(address, timeout=3.0) as client:
    client.sendall(b"AAA")
    client.sendall(b"BBB")
    client.sendall(b"CCC")
    time.sleep(0.6)
stop()

print("client made 3 separate sendall() calls: b'AAA', b'BBB', b'CCC'")
print("server's single recv() returned      :", result.get("got"))
print()
print("The boundaries are gone. Nothing in the protocol records where one")
print("message ended - and a receiver expecting 3-byte messages would be")
print("wrong the moment two of them arrived together.")

## Three ways to put the boundaries back

You have to add them yourself, in the layer above TCP. That is what a protocol *is*.

| Approach | How | Good for | Watch out for |
|---|---|---|---|
| **Length prefix** | send `[4-byte length][payload]` | binary, any content | must read the length fully first |
| **Delimiter** | end each message with `\n` | text, line protocols | the delimiter must not appear in the data |
| **Fixed size** | every message is exactly N bytes | telemetry, fixed records | wasteful; inflexible |

HTTP uses both of the first two: headers are newline-delimited, then `Content-Length` tells you how many body bytes follow.

### The helper everything needs

```
    def recv_exactly(sock, n):      <- recv(n) may return FEWER than n bytes.
        ...                            It is not an error. You must loop.
```

🔴 `recv(n)` returns *up to* `n` bytes. Treating its result as complete is the second most common TCP bug, right after assuming boundaries exist.

In [ ]:
import struct

HEADER = "!I"                              # network byte order, 4-byte unsigned
HEADER_SIZE = struct.calcsize(HEADER)


def recv_exactly(sock, n):
    """Read exactly n bytes, or raise. recv() alone may return fewer."""
    buf = bytearray()
    while len(buf) < n:
        chunk = sock.recv(n - len(buf))
        if not chunk:
            raise ConnectionError(
                f"peer closed after {len(buf)} of {n} bytes")
        buf += chunk
    return bytes(buf)


def send_message(sock, payload: bytes):
    sock.sendall(struct.pack(HEADER, len(payload)) + payload)


def recv_message(sock) -> bytes:
    (length,) = struct.unpack(HEADER, recv_exactly(sock, HEADER_SIZE))
    return recv_exactly(sock, length)


def framed_handler(conn):
    got = []
    for _ in range(3):
        got.append(recv_message(conn))
    result["got"] = got


result = {}
address, stop = serve(framed_handler)

with socket.create_connection(address, timeout=3.0) as client:
    for message in (b"AAA", b"BBB", b"CCC"):
        send_message(client, message)
    time.sleep(0.4)
stop()

print("with a 4-byte length prefix, the server read:")
print("   ", result.get("got"))
print()
print("on the wire, b'AAA' is:", struct.pack(HEADER, 3) + b"AAA")
print("                        ^^^^^^^^^^^^^^^^ length  ^^^^^ payload")

### The delimiter approach, and `makefile()`

For text protocols, a newline is simpler than a length prefix — and Python gives you it almost free. **`sock.makefile()`** wraps a socket in a buffered file object, so you can use `readline()` and iterate over it exactly like the files in **8.1**.

It handles the buffering that `recv_exactly` had to do by hand. The catch is the one in the table: if a message can itself contain a newline, this breaks — you need escaping, or a length prefix.

In [ ]:
def line_handler(conn):
    got = []
    # 'rb' so we read bytes; newline='' would matter for text mode
    with conn.makefile("rb") as stream:
        for raw in stream:                  # iterates line by line
            line = raw.rstrip(b"\n")
            if line == b"QUIT":
                break
            got.append(line.decode("utf-8"))
    result["got"] = got


result = {}
address, stop = serve(line_handler)

with socket.create_connection(address, timeout=3.0) as client:
    # deliberately batched into ONE sendall - the framing still works
    client.sendall(b"deploy api\ndeploy web\nrollback cache\nQUIT\n")
    time.sleep(0.4)
stop()

print("one sendall() containing four newline-delimited commands")
print("server parsed them as:")
for command in result.get("got", []):
    print("   ", command)
print("\n  makefile() gives a socket the same readline()/iteration as a file (8.1).")

## `send()` vs `sendall()`

```
    n = sock.send(data)      returns HOW MANY bytes it actually sent.
                             May be fewer than len(data). Your job to loop.

    sock.sendall(data)       loops for you. Returns None. Raises on failure.
```

🔴 **Use `sendall()`.** The failure mode of `send()` is nasty: on localhost with small payloads it always sends everything, so the bug never appears in development — then a real network with a full send buffer truncates a message in production.

The only reason to use `send()` is when you are managing partial writes deliberately, which in practice means non-blocking I/O (**11.4**).

In [ ]:
def count_bytes(conn):
    total = 0
    while True:
        chunk = conn.recv(65536)
        if not chunk:
            break
        total += len(chunk)
    result["total"] = total


result = {}
payload = b"x" * (4 * 1024 * 1024)          # 4 MB - big enough to matter
address, stop = serve(count_bytes, conn_timeout=10.0)

with socket.create_connection(address, timeout=10.0) as client:
    sent_by_send = client.send(payload)     # ONE call, no loop
    time.sleep(0.2)

time.sleep(0.5)
stop()

print(f"asked send() to write : {len(payload):,} bytes")
print(f"send() reported       : {sent_by_send:,} bytes")
print(f"server actually got   : {result.get('total', 0):,} bytes")

if sent_by_send < len(payload):
    print("\n🔴 send() wrote only part of it. sendall() would have looped.")
else:
    print("\nsend() happened to write it all - which is exactly the danger:")
    print("on loopback it usually does, so the bug hides until production.")
print("Use sendall() unless you are deliberately handling partial writes.")

## `SO_REUSEADDR`, `TIME_WAIT`, and "Address already in use"

Close a TCP server and try to restart it immediately, and you may get `OSError: [Errno 98] Address already in use` — even though nothing is running.

**Why.** After closing, the connection sits in **`TIME_WAIT`** for up to a couple of minutes. This is deliberate: it lets any straggling packets from the old connection die out before the same address pair can be reused, so they cannot be mistaken for part of a new connection.

**The fix.** `setsockopt(SOL_SOCKET, SO_REUSEADDR, 1)` before `bind()`, which says "binding is fine even if an old connection is still winding down". Every server in this notebook sets it.

> ⚠️ **`SO_REUSEADDR` does not mean the same thing on Windows.** There it allows two sockets to bind the *same* address, which is a hijacking risk rather than a convenience; Windows' closer equivalent is `SO_EXCLUSIVEADDRUSE`. On Linux and macOS, use `SO_REUSEADDR` on every server you write.

## When the other side disappears

| Exception | Meaning | Typical cause |
|---|---|---|
| `ConnectionRefusedError` | reached the machine, nothing listening | server not started, wrong port |
| `ConnectionResetError` | peer went away abruptly (RST) | process killed, container restarted |
| `BrokenPipeError` | you wrote to a connection the peer already closed | writing after they hung up |
| `TimeoutError` | nothing happened in time | slow peer, dropped packets, firewall |

All four inherit from `OSError`, so `except OSError` catches the lot — useful when you genuinely want to treat any network failure the same way (**6.1**).

🔴 Note the asymmetry: **writing to a closed connection may succeed.** The failure often appears on the *next* call, because your first write only reached the kernel buffer.

In [ ]:
def rude_hangup(conn):
    conn.recv(100)
    # Slam it shut with RST rather than a polite FIN
    conn.setsockopt(socket.SOL_SOCKET, socket.SO_LINGER,
                    struct.pack("ii", 1, 0))
    conn.close()


address, stop = serve(rude_hangup)

client = socket.create_connection(address, timeout=3.0)
client.sendall(b"are you there?")
time.sleep(0.4)

failed_on = None
for attempt in (1, 2, 3):
    try:
        client.sendall(b"still there?")
        print(f"  write {attempt}: succeeded")
    except (ConnectionResetError, BrokenPipeError) as exc:
        print(f"  write {attempt}: {type(exc).__name__}: {exc.strerror}")
        failed_on = attempt
        break
    except OSError as exc:
        print(f"  write {attempt}: {type(exc).__name__}: {exc}")
        failed_on = attempt
        break
    time.sleep(0.2)

client.close()
stop()

print()
if failed_on == 1:
    print("  Here the very first write failed: the RST had already arrived.")
elif failed_on:
    print(f"  Writes 1-{failed_on - 1} 'succeeded' - they only reached the kernel")
    print(f"  buffer. The failure surfaced on write {failed_on}.")
else:
    print("  All three writes 'succeeded' even though nobody was listening.")

print("  Which of those you get depends on timing and platform - and that")
print("  is the real lesson: a successful write proves only that the bytes")
print("  reached YOUR kernel. Only a reply proves the peer received them.")

In [ ]:
# ---- confirm nothing was left running ----
import threading

alive = [t.name for t in threading.enumerate() if t is not threading.main_thread()]
print("background threads still alive:", alive or "none")
print("\nEvery server above was stopped with stop(), and all of them were")
print("daemon threads - so even an abandoned one could not outlive the kernel.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Assuming one `sendall()` equals one `recv()`.** It does not. Frame your messages - length prefix, delimiter, or fixed size.
2. 🔴 **Treating `recv(n)` as returning exactly `n` bytes.** It returns *up to* `n`. Loop until you have what you need.
3. 🔴 **Using `send()` instead of `sendall()`.** On loopback it usually sends everything, so the truncation bug only shows up in production.
4. **Ignoring `recv()` returning `b""`.** That is end-of-stream. A loop that does not break on it spins forever.
5. **Leaving blocking calls with no timeout.** `accept()` and `recv()` block indefinitely by default.
6. **Forgetting `SO_REUSEADDR` on a server.** Restarting then fails with "Address already in use" until `TIME_WAIT` expires.
7. **Believing a successful write means delivery.** It reached your kernel buffer. Only a reply proves the peer got it.
8. **Binding a server to `0.0.0.0` by habit.** That exposes it on every interface. Use `127.0.0.1` unless you mean otherwise.
9. **Closing the listening socket instead of the connection socket** (or the reverse). `accept()` gives you a second socket; both need closing.

## Best Practices

- Frame every message explicitly. Decide on the scheme before writing any code.
- Wrap `recv` in a `recv_exactly` helper once, and use it everywhere.
- Use `sendall()` for writes and `socket.create_connection()` for clients.
- Set a timeout on every socket immediately after creating it.
- Use `with` on sockets, or `try/finally` - they hold file descriptors.
- For line-based text protocols, use `sock.makefile()` and let Python buffer.
- Set `SO_REUSEADDR` before `bind()` on every server (but read the Windows caveat).
- Catch `OSError` when any network failure should be handled the same way; catch the specific subclasses when they need different responses.
- In tests, bind to port 0 and read the assigned port from `getsockname()`.

## Practice Exercises

Try these before moving on.

1. Change `send_message` to use a 2-byte length prefix (`!H`). What is the largest message it can now carry, and what happens if you exceed it?
2. Add a `QUIT` command to the length-prefixed server so the client can end the conversation cleanly, instead of the server counting to three.
3. 🔴 Remove the `time.sleep(0.4)` from the framing demonstration and run it several times. Does the result change between runs? Explain why that makes this class of bug so hard to catch.
4. Write a client that sends a message whose declared length is larger than what it actually sends, then closes. Show that `recv_exactly` raises a clear error rather than hanging.
5. Build a tiny key-value server speaking `SET key value` / `GET key` over newline-framed text, using `makefile()`. Handle an unknown command.
6. Measure it: send 10,000 small messages with `sendall()` per message, then batched into groups of 100. Compare the time and explain the difference (look up Nagle's algorithm and `TCP_NODELAY`).
7. 🔴 Write a server that never sets a timeout, connect to it, and have the client connect but send nothing. Predict what the server does. Run it with a hard time limit so you can kill it.